# NB03 · SMILES and Linear Chemical Notations
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/solutions/nb03_smiles_and_linear_notations.ipynb)

**Estimated time:** ~75 min | **Difficulty:** ●●○

A comprehensive study of linear chemical notations, including the formal grammar of SMILES, canonicalization algorithms, randomized SMILES for data augmentation, stereochemical encoding (`@`, `@@`, `/`, `\`), InChI layers and InChIKeys, and cross-toolkit discrepancies on complex steroid scaffolds.

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Interpret and write** valid SMILES strings following formal grammar rules (organic subset, brackets, bonds, branches, ring closures, aromaticity).
2. **Decode and visualize** the 10 reference hand-drawing SMILES from the course lectures, verifying equivalence and multi-component representations.
3. **Analyze** the 6 complex drawn structures from the lecture slides, verifying their IUPAC nomenclature and chemical formulas.
4. **Implement** canonicalization and generate randomized SMILES for machine learning data augmentation.
5. **Encode and verify** tetrahedral and geometric stereochemistry (`@`, `@@`, `/`, `\`), linking them to CIP rules ($R/S$, $E/Z$).
6. **Compare** InChI layers and InChIKeys with SMILES, identifying how InChI handles tautomeric equivalence.
7. **Audit** cross-toolkit behavior between RDKit and Open Babel on challenging polycyclic steroid scaffolds.

## Prerequisites and Links

- NB00, NB01, NB02 completed.
- Lecture 01 (SMILES rules, bonds, branches, rings, problem in using software) and Lecture 02 (Linear notation, InChI, stereochemistry).

In [ ]:
import os, random, warnings
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors, inchi
from openbabel import openbabel as ob

SEED = 42
FULL_RUN = False
random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("NB03 setup completed.")

## 1. SMILES Grammar and Syntax Tour

The **Simplified Molecular Input Line Entry System (SMILES)**, developed by David Weininger in 1988, represents molecular graph topologies using ASCII strings:

1. **Organic Subset:** Elements without brackets assuming standard valences: `B`, `C`, `N`, `O`, `P`, `S`, `F`, `Cl`, `Br`, `I`.
2. **Bracket Atoms:** Required for charges, explicit hydrogens, non-standard isotopes, or inorganic atoms (e.g. `[NH4+]`, `[13CH4]`, `[Fe+2]`).
3. **Bonds:**
   - Single: `-` (optional, implied between uppercase letters)
   - Double: `=`
   - Triple: `#`
   - Aromatic: `:` (optional, implied between lowercase letters)
4. **Branches:** Specified inside parentheses `()`, nested arbitrarily deep.
5. **Ring Closures:** Indicated by appending integers to the connecting atoms: single digits `1`–`9` or two digits preceded by a percent sign `%10`–`%99`.
6. **Aromaticity:** Represented with lowercase atomic symbols (`c`, `n`, `o`, `s`, `p`).
7. **Disconnected Structures:** A period `.` separates individual disconnected molecules (e.g. salts or hydrates).

In [ ]:
# Demonstration of grammar elements
smiles_examples = [
    ("Aspirin (branching + aromatic)", "CC(=O)Oc1ccccc1C(=O)O"),
    ("Pyridine (aromatic heteroatom)", "c1ccncc1"),
    ("Acetonitrile (triple bond)", "CC#N"),
    ("Sodium acetate salt (disconnected)", "CC(=O)[O-].[Na+]"),
    ("Biphenyl (%10 ring closure)", "c1ccccc1-c2ccccc2")
]

for desc, smi in smiles_examples:
    mol = Chem.MolFromSmiles(smi)
    print(f"{desc:36s} -> Atoms: {mol.GetNumHeavyAtoms():2d}, Smiles: {Chem.MolToSmiles(mol)}")

## 2. The 10 Hand-Drawing SMILES from Slides

In Lecture 01, ten molecules were presented for manual drawing practice:

1. `CCO` (Ethanol)
2. `CC(=O)O` (Acetic acid)
3. `CC(=O)OCC.O` (Ethyl acetate + water)
4. `C=CCBr` (Allyl bromide)
5. `C#N` (Hydrogen cyanide / Acetonitrile)
6. `CCN(CC)CC` (Triethylamine)
7. `C(C(C(=O)O)N)O` (Serine)
8. `OC(=O)C(Br)(Cl)N` (2-amino-2-bromo-2-chloroacetic acid, form A)
9. `ClC(Br)(N)C(=O)O` (2-amino-2-bromo-2-chloroacetic acid, form B)
10. `O=C(O)C(N)(Br)C` (2-amino-2-bromopropanoic acid)

Let us load, render, and analyze them from `data/processed/d01_slide_draw_smiles.csv`.

In [ ]:
csv_draw_path = "../data/processed/d01_slide_draw_smiles.csv"
if not os.path.exists(csv_draw_path):
    csv_draw_path = "data/processed/d01_slide_draw_smiles.csv"

df_draw = pd.read_csv(csv_draw_path)
mols_draw = [Chem.MolFromSmiles(s) for s in df_draw["smiles"]]
legends_draw = [f"{row['name']} ({row['smiles']})" for _, row in df_draw.iterrows()]

Draw.MolsToGridImage(mols_draw, molsPerRow=2, subImgSize=(350, 180), legends=legends_draw)

### Key Observations from the 10 Molecules

1. **SMILES Invariance to Traversal Order:**
   Look at Molecule 8 (`OC(=O)C(Br)(Cl)N`) and Molecule 9 (`ClC(Br)(N)C(=O)O`). Both represent the exact same molecule (2-amino-2-bromo-2-chloroacetic acid), but written starting from different atoms. Canonicalization reduces them to identical canonical SMILES strings!
2. **Multi-Component Disconnected SMILES:**
   Molecule 3 (`CC(=O)OCC.O`) contains a period `.`, representing an equimolar mixture of ethyl acetate and water.

In [ ]:
m8 = Chem.MolFromSmiles("OC(=O)C(Br)(Cl)N")
m9 = Chem.MolFromSmiles("ClC(Br)(N)C(=O)O")
canon_8 = Chem.MolToSmiles(m8)
canon_9 = Chem.MolToSmiles(m9)

print(f"Molecule 8 canonical: {canon_8}")
print(f"Molecule 9 canonical: {canon_9}")
assert canon_8 == canon_9
print("✔ Canonical SMILES for items 8 and 9 are identical!")

m3 = Chem.MolFromSmiles("CC(=O)OCC.O")
frags = Chem.GetMolFrags(m3, asMols=True)
print(f"Molecule 3 fragments count: {len(frags)}")
print(f"Fragment 1: {Chem.MolToSmiles(frags[0])} (Ethyl acetate)")
print(f"Fragment 2: {Chem.MolToSmiles(frags[1])} (Water)")

## 3. The 6 Write-the-SMILES Molecules from Slides

The lecture slides featured 6 complex hand-drawn drug scaffolds. We have curated them in `data/processed/d01_slide_write_smiles.csv`:

1. **Triflupromazine analog:** Trifluoromethyl phenothiazine derivative
2. **Phenazone (Antipyrine):** Classic pyrazolone analgesic
3. **Retinol-type polyene alcohol:** Conjugated diterpenoid alcohol
4. **Histidine peptidomimetic:** Cyclohexylacetyl-histidine derivative
5. **Indole carboxylic acid derivative:** Dicarboxylic indole scaffold
6. **Dihydrobenzofuran alkyne ketone:** Terminal alkyne ketone

In [ ]:
csv_write_path = "../data/processed/d01_slide_write_smiles.csv"
if not os.path.exists(csv_write_path):
    csv_write_path = "data/processed/d01_slide_write_smiles.csv"

df_write = pd.read_csv(csv_write_path)
mols_write = [Chem.MolFromSmiles(s) for s in df_write["smiles"]]
legends_write = [f"{row['name']}" for _, row in df_write.iterrows()]

Draw.MolsToGridImage(mols_write, molsPerRow=2, subImgSize=(350, 220), legends=legends_write)

## 4. Canonicalization and Randomized SMILES

A molecular graph with $N$ atoms can theoretically be traversed in $N!$ different orders, yielding many synthetically valid SMILES strings for the exact same chemical structure.

**Canonicalization** applies a deterministic graph invariant numbering algorithm (such as Weininger's Morgan algorithm) to produce one unique string representation for database indexing.

### Randomized SMILES for Data Augmentation

In contrast, **randomized SMILES** (`doRandom=True`) deliberately shuffles the graph traversal order, generating diverse valid SMILES representations of the same molecule. This technique is widely used in modern molecular machine learning (chemical Transformers like ChemBERTa) as an effective data augmentation strategy.

In [ ]:
mol_ibuprofen = Chem.MolFromSmiles("CC(C)Cc1ccc(cc1)C(C)C(=O)O")
canon_smi = Chem.MolToSmiles(mol_ibuprofen, canonical=True)
print(f"Canonical SMILES: {canon_smi}")

print()
print("10 Unique Randomized SMILES for Ibuprofen:")
random_smiles = set()
while len(random_smiles) < 10:
    s = Chem.MolToSmiles(mol_ibuprofen, doRandom=True, canonical=False)
    random_smiles.add(s)

for i, s in enumerate(random_smiles, 1):
    print(f"{i:2d}: {s}")

## 5. Stereochemistry in Linear Notations

Stereoisomerism plays a profound role in pharmacology. Two enantiomers can exhibit drastically different therapeutic profiles:
- **Thalidomide:** $(R)$-enantiomer is a sedative/antiemetic; $(S)$-enantiomer is teratogenic, causing phocomelia.
- **Fenoterol:** $(R,R)$-enantiomer is a potent $\beta_2$-adrenoceptor agonist, whereas $(S,S)$ displays negligible potency.

### SMILES Stereochemical Syntax

1. **Tetrahedral Chirality:**
   - Looking from the neighbor preceding the chiral atom down toward the chiral center:
   - `@`: Anticlockwise order of the remaining three neighbors.
   - `@@`: Clockwise order of the remaining three neighbors.
2. **Double Bond Geometry ($E/Z$):**
   - Directional single bonds: `/` and `\`.
   - Trans double bond: `F/C=C/F` or `F\C=C\F`
   - Cis double bond: `F/C=C\F` or `F\C=C/F`

### Lecture Example: $(2R,3E,5E)$-2-hydroxy-3,5-heptadienenitrile

In [ ]:
smi_heptadiene = "N#C[C@@H](O)/C=C/C=C/C"
mol_hept = Chem.MolFromSmiles(smi_heptadiene)
Chem.AssignStereochemistry(mol_hept, cleanIt=True, force=True)

chiral_centers = Chem.FindMolChiralCenters(mol_hept, includeUnassigned=True)
print(f"Chiral centers detected: {chiral_centers}")

Draw.MolToImage(mol_hept, size=(400, 200))

## 6. InChI and InChIKey: Layered Chemical Architecture

The **IUPAC International Chemical Identifier (InChI)** is a non-proprietary identifier structured into hierarchical informational layers separated by forward slashes `/`:

1. **Main Layer:** Chemical formula (`/c` connectivity, `/h` hydrogen atoms).
2. **Charge Layer:** `/q` formal charge, `/p` proton count.
3. **Stereochemical Layer:** `/t` tetrahedral stereocenters, `/m` inverted/standard parity, `/s` stereo type.
4. **Isotopic Layer:** `/i` isotopic substitutions.
5. **Fixed-H Layer:** `/f` (for tautomeric distinction when mobile H invariance is disabled).

### InChIKey

The **InChIKey** is a fixed-length 27-character SHA-256 hash formatted as:
`XXXXXXXXXXXXXX-YYYYYYYYYY-Z`
- Block 1 (14 chars): Molecular connectivity skeleton.
- Block 2 (10 chars): Stereochemistry, isotopes, and protonation state.
- Block 3 (1 char): Identifier version and flag.

### Tautomeric Equivalence in InChI vs. SMILES

A fundamental distinction: standard InChI neutralizes mobile prototropic tautomers to the **same identifier**, whereas SMILES distinguishes localized bonds!

In [ ]:
# 2-pyridone vs 2-hydroxypyridine tautomerism
smi_keto = "O=C1NC=CC=C1"
smi_enol = "Oc1ccccn1"

mol_keto = Chem.MolFromSmiles(smi_keto)
mol_enol = Chem.MolFromSmiles(smi_enol)

print(f"SMILES keto: {Chem.MolToSmiles(mol_keto)}")
print(f"SMILES enol: {Chem.MolToSmiles(mol_enol)}")
print(f"Are SMILES identical? {Chem.MolToSmiles(mol_keto) == Chem.MolToSmiles(mol_enol)}")

inchi_keto = Chem.MolToInchi(mol_keto)
inchi_enol = Chem.MolToInchi(mol_enol)
print(f"InChI keto: {inchi_keto}")
print(f"InChI enol: {inchi_enol}")
print(f"Are standard InChIs identical? {inchi_keto == inchi_enol}")
assert inchi_keto == inchi_enol

## 7. 'Never Take It For Granted': Cross-Toolkit Discrepancies

Lecture 01 raised a crucial warning: different cheminformatics toolkits (e.g. RDKit vs. Open Babel) can produce diverging representations for complex polycyclic stereochemical scaffolds.

Let us test the complex steroid SMILES from the slides:
`[C@@]12([C@H](C3=C(C=CC=C3)C1)[C@@H](CC2)C(=O)C)C`

In [ ]:
steroid_smi = "[C@@]12([C@H](C3=C(C=CC=C3)C1)[C@@H](CC2)C(=O)C)C"

# RDKit parse
mol_rdkit = Chem.MolFromSmiles(steroid_smi)
rdkit_canon = Chem.MolToSmiles(mol_rdkit)
rdkit_centers = Chem.FindMolChiralCenters(mol_rdkit, includeUnassigned=True)
rdkit_inchi = Chem.MolToInchi(mol_rdkit)

# Open Babel parse
ob_mol = ob.OBMol()
ob_conv = ob.OBConversion()
ob_conv.SetInAndOutFormats("smi", "can")
ob_conv.ReadString(ob_mol, steroid_smi)
ob_canon = ob_conv.WriteString(ob_mol).strip()

ob_conv.SetOutFormat("inchi")
ob_inchi = ob_conv.WriteString(ob_mol).strip()

print(f"RDKit Canonical SMILES:      {rdkit_canon}")
print(f"Open Babel Canonical SMILES: {ob_canon}")
print(f"RDKit Chiral Centers:        {rdkit_centers}")
print(f"RDKit InChI:                 {rdkit_inchi}")
print(f"Open Babel InChI:            {ob_inchi}")

### Exercise 3.1

Hand-write SMILES strings for:
1. **Aspirin** (acetylsalicylic acid, formula $C_9H_8O_4$)
2. **Caffeine** (1,3,7-trimethylxanthine, formula $C_8H_{10}N_4O_2$)
3. **Sertraline** (chiral SSRI antidepressant with $(1S,4S)$ configuration: `CN[C@H]1CC[C@H](c2ccc(Cl)c(Cl)c2)c2ccccc21`, formula $C_{17}H_{17}Cl_2N$)

Return them as a dictionary `{'aspirin': smiles, 'caffeine': smiles, 'sertraline': smiles}`.

In [ ]:
def get_reference_smiles() -> dict[str, str]:
    return {
        "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
        "caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
        "sertraline": "CN[C@H]1CC[C@H](c2ccc(Cl)c(Cl)c2)c2ccccc21"
    }

In [ ]:
# Self-check for Exercise 3.1
smiles_dict = get_reference_smiles()
for name, smi in smiles_dict.items():
    mol = Chem.MolFromSmiles(smi)
    assert mol is not None, f"Failed to parse {name}"
assert Chem.MolFromSmiles(smiles_dict["aspirin"]).GetNumHeavyAtoms() == 13
assert Chem.MolFromSmiles(smiles_dict["caffeine"]).GetNumHeavyAtoms() == 14
assert Chem.MolFromSmiles(smiles_dict["sertraline"]).GetNumHeavyAtoms() == 20
print('✔ Exercise 3.1 passed!')

### Exercise 3.2

Write a function `detect_invalid_smiles(smiles_list: list[str]) -> list[str]` that inspects a list of SMILES strings and returns only the invalid strings that fail RDKit parsing.

In [ ]:
def detect_invalid_smiles(smiles_list: list[str]) -> list[str]:
    invalid = []
    for s in smiles_list:
        m = Chem.MolFromSmiles(s)
        if m is None:
            invalid.append(s)
    return invalid

In [ ]:
# Self-check for Exercise 3.2
test_candidates = ["CC(=O)O", "c1ccc1", "C1CC1", "C=C(C)C", "C1CCCC", "c1ccccc1[CH4]"]
invalids = detect_invalid_smiles(test_candidates)
# "c1ccc1" is anti-aromatic cyclobutadiene that fails kekulization
# "C1CCCC" has unclosed ring closure 1
assert "C1CCCC" in invalids
assert len(invalids) >= 2
print('✔ Exercise 3.2 passed!')

### Exercise 3.3

Write a function `count_stereocenters(smiles: str) -> dict[str, int]` that returns a dictionary with keys `'defined'` and `'unassigned'` counting the number of assigned vs unassigned chiral centers in a molecule using `Chem.FindMolChiralCenters`.

In [ ]:
def count_stereocenters(smiles: str) -> dict[str, int]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True)
    defined = sum(1 for idx, label in centers if label in ("R", "S"))
    unassigned = sum(1 for idx, label in centers if label == "?")
    return {"defined": defined, "unassigned": unassigned}

In [ ]:
# Self-check for Exercise 3.3
# Sertraline has 2 defined chiral centers
res_sert = count_stereocenters("CN[C@H]1CC[C@H](c2ccc(Cl)c(Cl)c2)c2ccccc21")
assert res_sert["defined"] == 2
assert res_sert["unassigned"] == 0

# Achiral ethanol
res_eth = count_stereocenters("CCO")
assert res_eth["defined"] == 0 and res_eth["unassigned"] == 0
print('✔ Exercise 3.3 passed!')

## Common Pitfalls

1. **Lowercase Aromatic Letters in Non-Aromatic Rings:** Writing `c1cccc1` specifies an anti-aromatic or radical cyclopentadienyl ring rather than cyclopentane (`C1CCCC1`).
2. **Loss of Stereochemistry on `RemoveHs`:** If a chiral center's configuration relies on explicit hydrogen ordering, stripping hydrogens inappropriately can invert or erase stereochemical parities.
3. **Canonical SMILES Variation Across Toolkits:** Do not compare canonical SMILES strings generated by RDKit directly against those generated by Open Babel or ChemAxon; each implements a distinct proprietary or toolkit-specific canonical ordering. Use InChI or InChIKey for cross-toolkit identity matching!

## Summary and Key Takeaways

- SMILES is a concise, human-readable ASCII notation for molecular graphs with intuitive rules for branching and ring closures.
- Alternative graph traversals generate different valid SMILES; canonicalization provides a unique string for database matching.
- Randomized SMILES provide a powerful data augmentation strategy for training deep learning chemical language models.
- Stereochemistry is explicitly captured using `@`/`@@` for tetrahedral centers and `/`/`\` for directional double bonds.
- InChI provides a standard layered architecture that inherently unifies prototropic tautomers; InChIKey provides a 27-character fixed hash for web lookup.
- Cross-toolkit validations should rely on InChI/InChIKey rather than canonical SMILES.

## References

- Weininger, D. SMILES, a chemical language and information system. *J. Chem. Inf. Comput. Sci.* **1988**, 28, 31–36.
- Heller, S. R. et al. InChI, the IUPAC International Chemical Identifier. *J. Cheminformatics* **2015**, 7, 23.
- Arús-Pous, J. et al. Randomized SMILES strings improve the quality of molecular generative models. *J. Cheminformatics* **2019**, 11, 71.

In [ ]:
# Self-test check
print("✔ NB03 checks passed")